In [15]:
import os

from libs.multilevel_squeme.coarsening import coarsen_chain
from libs.multilevel_squeme import partition_graph_metis

from libs.multilevel_squeme.qubo_formulation import build_qubo_from_graph
from libs.multilevel_squeme.quantum_annealing import (
    anneal_bipartition,
    lift_partition_to_finer
)
from libs.multilevel_squeme.qaoa import recursive_kway_qaoa
from libs.multilevel_squeme.quantum_annealing import recursive_kway_anneal

from libs.utils import (
    load_mtx,
    matrix_to_graph,
    summarize_generic
)

In [17]:
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))

k_matrix_path = os.path.join(data_dir, "hybrid_ma.classical.fem.matrix_k.mtx")
print("Using data_dir:", data_dir)

Using data_dir: /home/operation/reps/FEM-Graph-Partitioning-Toolkit-Multilevel-METIS-/data


In [18]:
# Load matrices
A_K = load_mtx(k_matrix_path, 'K')

if A_K is None:
    raise RuntimeError("Matrix load failed; check file paths printed above.")

print(f"Loaded: K | shape={A_K.shape}, nnz={A_K.nnz}")

Loaded: K | shape=(960, 960), nnz=30282
Loaded: K | shape=(960, 960), nnz=30282


In [19]:
# Build graphs from matrices
diag_K = A_K.diagonal()

# Map the matrix into the graph (only the K matrix is needed since we are partitioning most based on connectivity)
G_K = matrix_to_graph(A_K, symmetrize='sum', drop_diagonal=True, abs_weights=True, node_vweight='diag', diag=diag_K)

print(f"K: |V|={G_K.number_of_nodes()}, |E|={G_K.number_of_edges()}")

K: |V|=960, |E|=14661


In [58]:
# Parameters for the coarsening
coarsen_limit = 100
max_levels = 200
weight = 'weight'

# Parameters for the annealling
K_TARGET = 32                # number of parts desired
BALANCE_LAMBDA = 1
NUM_READS = 500

In [60]:
graphs, maps = coarsen_chain(G_K, trial_seed=42, coarsen_limit=coarsen_limit, max_levels=max_levels, weight=weight)

Gc, Gc_map = graphs[-1], maps[-1]
print(f"K: |V|={Gc.number_of_nodes()}, |E|={Gc.number_of_edges()}")

part_k_coarse_QA = recursive_kway_anneal(
    Gc, 
    K_TARGET,
    balance_weight=BALANCE_LAMBDA,
    num_reads=NUM_READS,
    choose_by='vweight'
)

# Lift back to original nodes
part_k_orig = lift_partition_to_finer(graphs, maps, part_k_coarse_QA)

# Summarize on original graph
summarize_generic(G_K, part_k_orig, f'K [quantum k={K_TARGET}]')

print("\n")

# Baseline METIS comparison for the same K_TARGET
metis_part_K = partition_graph_metis(G_K, nparts=K_TARGET, weight='weight', seed=42, verbose=0)
summarize_generic(G_K, metis_part_K, f"METIS K [{K_TARGET}]")

K: |V|=69, |E|=548
K [quantum k=32]: cut=36479.8525, parts=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31], per=[1129.4869594736917, 1107.3933673179356, 1110.877415168112, 973.0297549479416, 1150.50584234722, 1026.0449654096467, 1019.4966574954029, 911.9327677520997, 1056.6678741451422, 1337.526627445051, 1055.6936728216206, 1070.0294586206714, 1014.7553679067283, 1092.8527645186518, 761.3741261215536, 980.5148539694788, 1090.9002392789976, 1161.9423439630325, 1021.9522645497517, 1095.864775612167, 1057.7790042384859, 976.1686124477379, 1009.7989248520497, 1032.4079594743334, 1072.480567760169, 985.3898885478123, 1178.9376026352136, 1074.0245990933486, 748.8050150974158, 1103.9545202861484, 933.6254989866169, 1303.6506815576258], total=33645.8650
Per difference: 588.7216123476352 Max per: 1337.526627445051 Min per: 748.8050150974158


METIS K [32]: cut=27546.6626, parts=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,